# Air_Tourism_Collector (converted from .py)

- This notebook was automatically generated from `Air_Tourism_Collector.py`.
- Run the main collector cell first.
- Then run the debug cell to verify the ODP date range (min/max) and monthly counts.


In [3]:
"""
air_tourism_collector.py
------------------------
세 개의 노트북(공공데이터포털 항공통계 API / AirPortal 엑셀 다운로드 / KAC(airport.co.kr) 크롤링)
+ (옵션) FDR(유가/환율), BOK ECOS(여행비 지출전망 CSI)
를 통합하여 **하나의 long-format 데이터프레임**으로 만드는 모듈입니다.

출력 기본 스키마 (권장 long_format)
- date        : pandas.Timestamp (월말 기준)
- indicator   : 지표명(소스별 prefix 포함)
- value       : float
- source      : 데이터 출처
- entity_type : airline / (blank)
- entity_code : 항공사 코드 등
- entity_name : 항공사 한글명 등
- extra_json  : 원본에서 남기고 싶은 보조정보(JSON string)

필요 패키지:
- pandas, requests, openpyxl, python-dateutil
- (옵션) FinanceDataReader (유가/환율)
"""

from __future__ import annotations

import json
import time
from dataclasses import dataclass
from datetime import datetime
from io import BytesIO
from typing import Dict, Iterable, List, Optional, Tuple

import pandas as pd
import requests
from dateutil.relativedelta import relativedelta


# =========================================================
# 공통 유틸
# =========================================================

def month_range(start_yyyymm: str, end_yyyymm: str) -> Iterable[str]:
    """YYYYMM 포함 범위 generator"""
    cur = datetime.strptime(start_yyyymm, "%Y%m")
    end = datetime.strptime(end_yyyymm, "%Y%m")
    while cur <= end:
        yield cur.strftime("%Y%m")
        cur += relativedelta(months=1)


def to_month_end_ts(yyyymm: str) -> pd.Timestamp:
    """YYYYMM -> 월말 Timestamp"""
    return pd.to_datetime(yyyymm, format="%Y%m") + pd.offsets.MonthEnd(0)


def _coerce_numeric(s: pd.Series) -> pd.Series:
    """쉼표/공백 제거 후 numeric"""
    return pd.to_numeric(
        s.astype(str).str.replace(",", "", regex=False).str.strip(),
        errors="coerce"
    )


def _safe_json_dumps(obj) -> str:
    try:
        return json.dumps(obj, ensure_ascii=False)
    except Exception:
        return ""


def _standard_long(
    df: pd.DataFrame,
    *,
    date_col: str,
    value_col: str,
    indicator: str,
    source: str,
    entity_type: str = "",
    entity_code_col: Optional[str] = None,
    entity_name_col: Optional[str] = None,
    extras: Optional[List[str]] = None,
) -> pd.DataFrame:
    """단일 지표 df -> 표준 long-format"""
    out = pd.DataFrame()
    out["date"] = pd.to_datetime(df[date_col])
    out["indicator"] = indicator
    out["value"] = pd.to_numeric(df[value_col], errors="coerce")
    out["source"] = source
    out["entity_type"] = entity_type

    if entity_code_col and entity_code_col in df.columns:
        out["entity_code"] = df[entity_code_col].astype(str)
    else:
        out["entity_code"] = ""

    if entity_name_col and entity_name_col in df.columns:
        out["entity_name"] = df[entity_name_col].astype(str)
    else:
        out["entity_name"] = ""

    # extra_json
    if extras:
        extra_obj = df[extras].to_dict(orient="records")
        out["extra_json"] = [_safe_json_dumps(x) for x in extra_obj]
    else:
        out["extra_json"] = ""

    return out


def melt_numeric_to_long(
    df: pd.DataFrame,
    *,
    id_cols: List[str],
    date_col: str,
    source: str,
    indicator_prefix: str,
    entity_type: str = "",
    entity_code_col: Optional[str] = None,
    entity_name_col: Optional[str] = None,
    keep_extra_cols: Optional[List[str]] = None,
) -> pd.DataFrame:
    """
    df의 numeric 컬럼들을 자동 감지해 long으로 변환.
    - id_cols: melt의 id_vars
    - date_col: 최종 long에 들어갈 날짜 컬럼(이미 Timestamp 권장)
    - indicator: indicator_prefix + 컬럼명
    """
    keep_extra_cols = keep_extra_cols or []
    id_vars = list(dict.fromkeys(id_cols + [date_col]))  # 중복 제거 유지

    # 후보 컬럼: id_vars/엔티티/extra 제외 후 numeric 변환 가능한 컬럼
    exclude = set(id_vars + keep_extra_cols)
    if entity_code_col:
        exclude.add(entity_code_col)
    if entity_name_col:
        exclude.add(entity_name_col)

    candidates = [c for c in df.columns if c not in exclude]

    # 숫자 컬럼 판정: 변환 후 유효값이 어느 정도 있는지
    numeric_cols = []
    for c in candidates:
        tmp = _coerce_numeric(df[c])
        if tmp.notna().sum() >= max(1, int(0.05 * len(tmp))):  # 5% 이상 숫자면 채택
            numeric_cols.append(c)
            df[c] = tmp

    if not numeric_cols:
        # 아무것도 못 찾으면, 그대로 빈 df 반환
        return pd.DataFrame(columns=["date", "indicator", "value", "source", "entity_type", "entity_code", "entity_name", "extra_json"])

    # melt할 id_vars를 DataFrame에 실제 존재하는 컬럼만 포함
    melt_id_vars = id_vars.copy()
    if entity_code_col and entity_code_col in df.columns:
        melt_id_vars.append(entity_code_col)
    if entity_name_col and entity_name_col in df.columns:
        melt_id_vars.append(entity_name_col)
    for col in keep_extra_cols:
        if col in df.columns:
            melt_id_vars.append(col)

    # 중복 제거
    melt_id_vars = list(dict.fromkeys(melt_id_vars))

    m = df.melt(id_vars=melt_id_vars,
                value_vars=numeric_cols,
                var_name="field",
                value_name="value")

    m = m.rename(columns={date_col: "date"})
    m["indicator"] = m["field"].apply(lambda x: f"{indicator_prefix}{str(x).strip()}")
    m["source"] = source
    m["entity_type"] = entity_type
    if entity_code_col:
        m["entity_code"] = m[entity_code_col].astype(str)
    else:
        m["entity_code"] = ""
    if entity_name_col:
        m["entity_name"] = m[entity_name_col].astype(str)
    else:
        m["entity_name"] = ""

    # extra_json
    extras = keep_extra_cols + id_cols
    extras = [c for c in extras if c in m.columns and c not in ["date", "indicator", "value", "source", "entity_type", "entity_code", "entity_name", "field"]]
    if extras:
        m["extra_json"] = [
            _safe_json_dumps({k: r[k] for k in extras})
            for r in m[extras].to_dict(orient="records")
        ]
    else:
        m["extra_json"] = ""

    return m[["date", "indicator", "value", "source", "entity_type", "entity_code", "entity_name", "extra_json"]].copy()


# =========================================================
# (A) 공공데이터포털 - 인천공항공사(B551177) 항공사별 통계 API
#   - notebook: 2_Air_Tourism.ipynb 의 AviationStatsByAirline 부분
# =========================================================

@dataclass
class ODPIncheonAirlineStatsConfig:
    service_key: str
    start_yyyymm: str = "202201"
    end_yyyymm: Optional[str] = None  # None이면 현재월까지
    airlines: Optional[Dict[str, str]] = None  # {"KE":"대한항공", ...}
    timeout: int = 30
    sleep_sec: float = 0.1


class ODPIncheonAirlineStatsClient:
    BASE = "https://apis.data.go.kr/B551177/AviationStatsByAirline"

    def __init__(self, cfg: ODPIncheonAirlineStatsConfig):
        self.cfg = cfg

    @staticmethod
    def _normalize_json(js):
        if isinstance(js, list):
            return js[0] if js else {}
        return js if isinstance(js, dict) else {}

    def _extract_items(self, js) -> List[dict]:
        js = self._normalize_json(js)
        resp = self._normalize_json(js.get("response", js))
        body = self._normalize_json(resp.get("body", resp))
        if not isinstance(body, dict):
            return []
        items_block = body.get("items", None)

        if isinstance(items_block, dict):
            items = items_block.get("item", [])
        elif isinstance(items_block, list):
            items = items_block
        else:
            items = body.get("item", [])

        if items is None:
            items = []
        if isinstance(items, dict):
            items = [items]
        if isinstance(items, list) and len(items) == 1 and isinstance(items[0], dict) and "item" in items[0]:
            nested = items[0]["item"]
            if isinstance(nested, list):
                items = nested
            elif isinstance(nested, dict):
                items = [nested]
        return items

    def _call(self, endpoint: str, params: dict) -> dict:
        url = f"{self.BASE}/{endpoint}"
        r = requests.get(url, params=params, timeout=self.cfg.timeout)
        try:
            r.raise_for_status()
        except Exception as e:
            raise RuntimeError(f"HTTP error: {e} / url={r.url} / body={r.text[:300]}")
        try:
            return r.json()
        except Exception:
            raise RuntimeError(f"Non-JSON response (maybe XML error). url={r.url} / body={r.text[:400]}")

    def _fetch_one(self, endpoint: str, yyyymm: str) -> pd.DataFrame:
        params = {
            "serviceKey": self.cfg.service_key,
            "from_month": yyyymm,
            "to_month": yyyymm,
            "type": "json",
        }
        js = self._call(endpoint, params=params)
        items = self._extract_items(js)
        df = pd.DataFrame(items)
        if df.empty:
            return df
        df["yyyymm"] = yyyymm
        df["date"] = to_month_end_ts(yyyymm)
        return df

    def fetch(self) -> pd.DataFrame:
        airlines = self.cfg.airlines or {
            "KE": "대한항공",
            "OZ": "아시아나항공",
            "7C": "제주항공",
            "AA": "아메리칸항공",
        }
        end_yyyymm = self.cfg.end_yyyymm or datetime.today().strftime("%Y%m")

        frames = []
        for yyyymm in month_range(self.cfg.start_yyyymm, end_yyyymm):
            df_p = self._fetch_one("getTotalNumberOfPassenger", yyyymm)
            df_c = self._fetch_one("getTotalTonsOfCargo", yyyymm)

            for df in (df_p, df_c):
                if df is None or df.empty:
                    continue
                if "airlineCode" not in df.columns:
                    continue
                df = df[df["airlineCode"].isin(airlines.keys())].copy()
                df["airlineName_kr"] = df["airlineCode"].map(airlines)
                frames.append(df)

            time.sleep(self.cfg.sleep_sec)

        if not frames:
            return pd.DataFrame()
        return pd.concat(frames, ignore_index=True)


    @staticmethod
    def to_long(df_raw: pd.DataFrame) -> pd.DataFrame:
        """
        원본 컬럼(예상):
        - passenger: passenger/arrPassenger/depPassenger
        - cargo: baggage/arrBaggage/depBaggage (톤)
        """
        if df_raw is None or df_raw.empty:
            return pd.DataFrame(columns=["date", "indicator", "value", "source", "entity_type", "entity_code", "entity_name", "extra_json"])

        df = df_raw.copy()

        # 숫자 변환
        for c in ["arrPassenger", "depPassenger", "passenger", "arrBaggage", "depBaggage", "baggage"]:
            if c in df.columns:
                df[c] = _coerce_numeric(df[c])

        long_parts: List[pd.DataFrame] = []

        # passenger set
        pax_cols = [c for c in ["passenger", "arrPassenger", "depPassenger"] if c in df.columns]
        if pax_cols:
            tmp = df[["date", "airlineCode", "airlineName_kr"] + pax_cols].copy()
            tmp_long = melt_numeric_to_long(
                tmp,
                id_cols=[],
                date_col="date",
                source="ODP_B551177",
                indicator_prefix="icn_airline_pax_",
                entity_type="airline",
                entity_code_col="airlineCode",
                entity_name_col="airlineName_kr",
            )
            long_parts.append(tmp_long)

        # cargo set
        cargo_cols = [c for c in ["baggage", "arrBaggage", "depBaggage"] if c in df.columns]
        if cargo_cols:
            tmp = df[["date", "airlineCode", "airlineName_kr"] + cargo_cols].copy()
            tmp_long = melt_numeric_to_long(
                tmp,
                id_cols=[],
                date_col="date",
                source="ODP_B551177",
                indicator_prefix="icn_airline_cargo_tons_",
                entity_type="airline",
                entity_code_col="airlineCode",
                entity_name_col="airlineName_kr",
            )
            long_parts.append(tmp_long)

        if not long_parts:
            return pd.DataFrame(columns=["date", "indicator", "value", "source", "entity_type", "entity_code", "entity_name", "extra_json"])

        out = pd.concat(long_parts, ignore_index=True)
        return out.dropna(subset=["value"]).sort_values(["date", "indicator", "entity_code"]).reset_index(drop=True)


# =========================================================
# (B) AirPortal - 엑셀 다운로드(월별)
#   - notebook: 2_Air_Tourism_Air_Portal.ipynb
# =========================================================

@dataclass
class AirPortalConfig:
    start_year: int = 2015
    start_month: int = 1
    end_year: int = 2025
    end_month: int = 12
    delay_sec: float = 1.0
    timeout: int = 30


class AirPortalClient:
    BASE = "https://www.airportal.go.kr"
    EXCEL_ENDPOINT = "/stats/transport/getDetailedAirTransportStats1Excel.do"

    def __init__(self, cfg: AirPortalConfig):
        self.cfg = cfg

    def download_one_month(self, year: int, month: int) -> Optional[pd.DataFrame]:
        excel_url = f"{self.BASE}{self.EXCEL_ENDPOINT}"
        yyyymm = f"{year}{month:02d}"

        params = {
            "last_yearmonth": yyyymm,
            "this_yearmonth": yyyymm,
            "pass_gubun": "4",
            "carge_gubun": "total",
            "sn_gubun": "total",
            "airline_gubun": "total",
            "di_gubun": "total",
            "pyn_gubun": "total",
            "arvl_type": "total",
        }

        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
            "Accept": "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
            "Content-Type": "application/json;charset=UTF-8",
            "Referer": f"{self.BASE}/stats/transport/chartDetail.do",
        }

        try:
            r = requests.post(excel_url, json=params, headers=headers, timeout=self.cfg.timeout)
            if r.status_code != 200:
                print(f"[AIRPORTAL] {year}-{month:02d} fail: HTTP {r.status_code}")
                return None

            df = pd.read_excel(BytesIO(r.content), engine="openpyxl")
            df = df.dropna(axis=1, how="all")
            df = df.loc[:, ~df.columns.astype(str).str.contains("^Unnamed", na=False)]

            df["year"] = year
            df["month"] = month
            df["yyyymm"] = yyyymm
            df["date"] = to_month_end_ts(yyyymm)

            print(f"[AIRPORTAL] ✓ {year}-{month:02d} rows={len(df)}")
            return df

        except Exception as e:
            print(f"[AIRPORTAL] {year}-{month:02d} error: {e}")
            return None

    def download_period(self) -> pd.DataFrame:
        all_data = []
        y, m = self.cfg.start_year, self.cfg.start_month

        total_months = (self.cfg.end_year - self.cfg.start_year) * 12 + (self.cfg.end_month - self.cfg.start_month) + 1
        processed = 0

        while (y < self.cfg.end_year) or (y == self.cfg.end_year and m <= self.cfg.end_month):
            processed += 1
            df = self.download_one_month(y, m)
            if df is not None and not df.empty:
                all_data.append(df)

            m += 1
            if m > 12:
                m = 1
                y += 1

            if processed < total_months:
                time.sleep(self.cfg.delay_sec)

        if not all_data:
            return pd.DataFrame()
        return pd.concat(all_data, ignore_index=True)

    @staticmethod
    def to_long(df_raw: pd.DataFrame) -> pd.DataFrame:
        """
        AirPortal 엑셀은 컬럼/헤더가 변동될 수 있어,
        1) 날짜(yyyymm/date/year/month) 컬럼을 id로 둠
        2) 숫자로 변환 가능한 컬럼을 전부 melt
        3) 나머지 텍스트 컬럼은 extra_json에 저장

        ✅ (수정) AIRPORTAL은 항공사코드 컬럼이 없는 경우가 많아,
        '항공사명'의 괄호 코드(예: 대한항공(KAL))를 entity_code로 추출해 중복 제거가 깨지지 않게 함.
        """
        if df_raw is None or df_raw.empty:
            return pd.DataFrame(columns=[
                "date", "indicator", "value", "source",
                "entity_type", "entity_code", "entity_name", "extra_json"
            ])

        df = df_raw.copy()

        # date 컬럼 확보
        if "date" not in df.columns:
            if "yyyymm" in df.columns:
                df["date"] = df["yyyymm"].apply(to_month_end_ts)
            elif "년월" in df.columns:
                df["date"] = pd.to_datetime(df["년월"].astype(str) + "-01") + pd.offsets.MonthEnd(0)
            else:
                raise ValueError("AirPortal data에 date/yyyymm/년월 컬럼이 없습니다.")

        # 엔티티(항공사 등)가 있는지 자동 탐색 (자주 쓰는 후보)
        entity_code_col = None
        entity_name_col = None

        for c in ["항공사코드", "항공사코드(2)", "AIRLINE_CD", "AIRLINE_CODE", "airlineCode"]:
            if c in df.columns:
                entity_code_col = c
                break

        for c in ["항공사", "항공사명", "AIRLINE_NM", "AIRLINE_NAME", "airlineName"]:
            if c in df.columns:
                entity_name_col = c
                break

        # ✅ 대안 1: 항공사코드 컬럼이 없으면 '항공사명'에서 코드 추출해서 entity_code 채우기
        if entity_code_col is None and entity_name_col is not None:
            # 예: "대한항공(KAL)" -> code="KAL", clean_name="대한항공"
            extracted_code = (
                df[entity_name_col]
                .astype(str)
                .str.extract(r"\(([^)]+)\)\s*$")[0]
            )
            df["_entity_code"] = extracted_code.fillna(df[entity_name_col].astype(str))
            df["_entity_name_clean"] = (
                df[entity_name_col]
                .astype(str)
                .str.replace(r"\s*\([^)]+\)\s*$", "", regex=True)
            )

            entity_code_col = "_entity_code"
            entity_name_col = "_entity_name_clean"

        # extra로 보낼 텍스트 컬럼(숫자 아닌 것들)
        id_cols = [c for c in ["year", "month", "yyyymm", "년월", "년도", "월"] if c in df.columns]
        text_cols = []
        for c in df.columns:
            if c in id_cols or c in ["date", entity_code_col, entity_name_col]:
                continue
            tmp = _coerce_numeric(df[c])
            if tmp.notna().sum() < max(1, int(0.05 * len(tmp))):
                text_cols.append(c)

        long_df = melt_numeric_to_long(
            df,
            id_cols=id_cols + text_cols,   # text_cols도 유지해서 extra_json에 들어가게
            date_col="date",
            source="AIRPORTAL",
            indicator_prefix="airportal_",
            entity_type="airline" if (entity_code_col or entity_name_col) else "",
            entity_code_col=entity_code_col,
            entity_name_col=entity_name_col,
            keep_extra_cols=text_cols,
        )

        # ✅ (선택) 임시 컬럼이 extra_json으로 들어가는 게 싫으면 여기서 제거하고 싶겠지만,
        # melt_numeric_to_long 내부에서 extra_json 생성 로직이 있으므로 보통은 그대로 둬도 무방합니다.

        return (
            long_df
            .dropna(subset=["value"])
            .sort_values(["date", "indicator", "entity_code"])
            .reset_index(drop=True)
        )


# =========================================================
# (C) KAC (airport.co.kr) - 크롤링(ajax JSON)
#   - notebook: 2_Air_Tourism_KAC_Crawling.ipynb
# =========================================================

@dataclass
class KACConfig:
    start_yyyymm: str = "202001"
    end_yyyymm: str = "202512"
    pass_type: str = "C4101"  # 유임여객
    cago_type: str = "C4201"  # 화물
    sleep_sec: float = 0.3
    timeout: int = 30


class KACClient:
    BASE = "https://www.airport.co.kr"
    ENTRY_URL = BASE + "/www/cms/frCon/index.do?MENU_ID=1250"
    LIST_URL = BASE + "/www/ajaxf/frFlightStatsSvc/airLineStatsList.do"

    def __init__(self, cfg: KACConfig):
        self.cfg = cfg

    def _new_session(self) -> requests.Session:
        s = requests.Session()
        s.headers.update({
            "User-Agent": "Mozilla/5.0",
            "Referer": self.ENTRY_URL,
        })
        s.get(self.ENTRY_URL, timeout=self.cfg.timeout)  # 쿠키 세팅
        return s

    def fetch_one_month(self, session: requests.Session, yyyymm: str) -> pd.DataFrame:
        payload = {
            "ST_YY": yyyymm[:4],
            "ST_MM": yyyymm[4:],
            "EN_YY": yyyymm[:4],
            "EN_MM": yyyymm[4:],
            "PASS_TYPE": self.cfg.pass_type,
            "CAGO_TYPE": self.cfg.cago_type,
        }
        r = session.post(self.LIST_URL, data=payload, timeout=self.cfg.timeout)
        r.raise_for_status()
        data = r.json()  # list[dict]
        if not data:
            return pd.DataFrame()

        df = pd.json_normalize(data)
        df["yyyymm"] = yyyymm
        df["date"] = to_month_end_ts(yyyymm)
        return df

    def fetch(self) -> pd.DataFrame:
        session = self._new_session()
        frames = []
        for yyyymm in month_range(self.cfg.start_yyyymm, self.cfg.end_yyyymm):
            try:
                print(f"[KAC] fetch {yyyymm}")
                df_m = self.fetch_one_month(session, yyyymm)
                if not df_m.empty:
                    frames.append(df_m)
                time.sleep(self.cfg.sleep_sec)
            except Exception as e:
                print(f"[KAC] WARN {yyyymm} failed: {e}")

        if not frames:
            return pd.DataFrame()
        return pd.concat(frames, ignore_index=True)

    @staticmethod
    def normalize_nested_data(df_raw: pd.DataFrame) -> pd.DataFrame:
        """
        KAC 응답에서 'data' 컬럼이 list[dict]로 들어오는 케이스를
        notebook 방식처럼 explode+json_normalize로 평탄화.
        """
        if df_raw is None or df_raw.empty:
            return pd.DataFrame()

        df = df_raw.copy()
        if "data" not in df.columns:
            return df  # 이미 평탄화 되었을 수도

        left = df.explode("data").drop(columns=["data"]).reset_index(drop=True)
        right = pd.json_normalize(df.explode("data")["data"])
        out = pd.concat([left.reset_index(drop=True), right.reset_index(drop=True)], axis=1)
        return out

    @staticmethod
    def to_long(df_raw: pd.DataFrame) -> pd.DataFrame:
        """
        KAC 데이터는 컬럼명이 상황에 따라 다릅니다.
        - date/yyyymm/항공사 식별 컬럼 후보(A_AIRLINE, A_AIRKOR 등)를 자동 탐색
        - 숫자 변환 가능한 컬럼을 전부 melt
        """
        if df_raw is None or df_raw.empty:
            return pd.DataFrame(columns=["date", "indicator", "value", "source", "entity_type", "entity_code", "entity_name", "extra_json"])

        df = KACClient.normalize_nested_data(df_raw)

        # 엔티티 후보
        entity_code_col = None
        entity_name_col = None
        for c in ["A_AIRLINE", "airlineCode", "AIRLINE_CD", "항공사코드"]:
            if c in df.columns:
                entity_code_col = c
                break
        for c in ["A_AIRKOR", "airlineName", "AIRLINE_NM", "항공사", "항공사명"]:
            if c in df.columns:
                entity_name_col = c
                break

        # date 컬럼 확보
        if "date" not in df.columns:
            if "yyyymm" in df.columns:
                df["date"] = df["yyyymm"].apply(to_month_end_ts)
            else:
                raise ValueError("KAC data에 date/yyyymm 컬럼이 없습니다.")

        # 텍스트 보조컬럼을 extra_json로 보낼지 후보 선정
        id_cols = [c for c in ["yyyymm"] if c in df.columns]
        # KAC에 자주 있는 구분 컬럼 후보
        keep_text = [c for c in ["PASS_TYPE", "CAGO_TYPE", "LINE_TYPE", "RAGUL_TYPE", "USE_TYPE", "AL_TYPE"] if c in df.columns]

        long_df = melt_numeric_to_long(
            df,
            id_cols=id_cols + keep_text,
            date_col="date",
            source="KAC_AIRPORT",
            indicator_prefix="kac_",
            entity_type="airline" if (entity_code_col or entity_name_col) else "",
            entity_code_col=entity_code_col,
            entity_name_col=entity_name_col,
            keep_extra_cols=keep_text,
        )

        return long_df.dropna(subset=["value"]).sort_values(["date", "indicator", "entity_code"]).reset_index(drop=True)


# =========================================================
# (D) (옵션) FDR: 유가/환율, BOK ECOS: 여행비 CSI
#   - notebook: 2_Air_Tourism.ipynb 의 FDR + ECOS 부분
# =========================================================

def fetch_oil_fx_fdr(start: str = "2010-01-01") -> pd.DataFrame:
    """
    FinanceDataReader 기반 유가 + 환율 수집
    - indicator: oil_wti, oil_brent, usdkrw
    """
    try:
        import FinanceDataReader as fdr  # type: ignore
    except Exception as e:
        raise ImportError("FinanceDataReader가 설치되어 있지 않습니다. (pip install finance-datareader)") from e

    data = []
    series_map = {
        "oil_wti": "CL=F",
        "oil_brent": "BZ=F",
        "usdkrw": "USD/KRW",
    }

    for indicator, symbol in series_map.items():
        df = fdr.DataReader(symbol, start).reset_index()
        if "Date" in df.columns:
            df = df.rename(columns={"Date": "date"})
        elif "date" not in df.columns:
            df = df.rename(columns={df.columns[0]: "date"})

        if "Close" in df.columns:
            value_col = "Close"
        elif "Price" in df.columns:
            value_col = "Price"
        else:
            raise ValueError(f"[FDR] 가격 컬럼 없음: {symbol}")

        tmp = df[["date", value_col]].rename(columns={value_col: "value"})
        tmp["indicator"] = indicator
        tmp["source"] = "FDR"
        tmp["entity_type"] = ""
        tmp["entity_code"] = ""
        tmp["entity_name"] = ""
        tmp["extra_json"] = ""
        data.append(tmp)

    out = pd.concat(data, ignore_index=True)
    out["date"] = pd.to_datetime(out["date"])
    return out[["date", "indicator", "value", "source", "entity_type", "entity_code", "entity_name", "extra_json"]].copy()


def ecos_stat_search(
    api_key: str,
    stat_code: str,
    cycle: str,
    start_date: str,
    end_date: str,
    item_code1: str = "",
    item_code2: str = "",
    item_code3: str = "",
    lang: str = "kr",
    timeout: int = 30,
) -> pd.DataFrame:
    """
    BOK ECOS StatisticSearch 호출 (JSON)
    - cycle: D/M/Q/A
    - start_date/end_date: M이면 YYYYMM
    """
    base = "https://ecos.bok.or.kr/api"
    url = "/".join([
        base, "StatisticSearch",
        api_key, "json", lang,
        "1", "100000",
        stat_code, cycle, start_date, end_date,
        item_code1, item_code2, item_code3
    ])

    r = requests.get(url, timeout=timeout)
    r.raise_for_status()
    js = r.json()
    rows = js.get("StatisticSearch", {}).get("row", [])
    df = pd.DataFrame(rows)
    if df.empty:
        return df

    df = df.rename(columns={
        "TIME": "date",
        "DATA_VALUE": "value",
        "STAT_CODE": "stat_code",
        "STAT_NAME": "stat_name",
        "ITEM_CODE1": "item_code1",
        "ITEM_NAME1": "item_name1",
        "ITEM_CODE2": "item_code2",
        "ITEM_NAME2": "item_name2",
        "ITEM_CODE3": "item_code3",
        "ITEM_NAME3": "item_name3",
        "UNIT_NAME": "unit",
    })
    df["value"] = pd.to_numeric(df["value"], errors="coerce")

    if cycle == "M":
        df["date"] = pd.to_datetime(df["date"], format="%Y%m") + pd.offsets.MonthEnd(0)
    else:
        df["date"] = df["date"].astype(str)

    return df


def fetch_travel_spending_expectation_csi_total(
    api_key: str,
    start_date: str = "200809",
    end_date: str = "202512",
) -> pd.DataFrame:
    """
    소비자동향조사 - 여행비 지출전망 CSI (전체)
    indicator: csi_travel_spending_expectation_total
    """
    df = ecos_stat_search(
        api_key=api_key,
        stat_code="511Y002",
        cycle="M",
        start_date=start_date,
        end_date=end_date,
        item_code1="FMCCD",
        item_code2="99988",
    )
    if df.empty:
        return pd.DataFrame(columns=["date", "indicator", "value", "source", "entity_type", "entity_code", "entity_name", "extra_json"])

    out = df[["date", "value", "stat_code", "stat_name", "unit", "item_code1", "item_name1", "item_code2", "item_name2"]].copy()
    out["indicator"] = "csi_travel_spending_expectation_total"
    out["source"] = "BOK_ECOS"
    out["entity_type"] = ""
    out["entity_code"] = ""
    out["entity_name"] = ""
    out["extra_json"] = [_safe_json_dumps({
        "stat_code": r["stat_code"],
        "stat_name": r["stat_name"],
        "unit": r["unit"],
        "item_code1": r["item_code1"],
        "item_name1": r["item_name1"],
        "item_code2": r["item_code2"],
        "item_name2": r["item_name2"],
    }) for r in out.to_dict(orient="records")]
    out["value"] = pd.to_numeric(out["value"], errors="coerce")
    return out[["date", "indicator", "value", "source", "entity_type", "entity_code", "entity_name", "extra_json"]].sort_values("date").reset_index(drop=True)


# =========================================================
# 통합 수집 함수
# =========================================================

@dataclass
class AirTourismCollectorConfig:
    # (필수) ODP Incheon API key (Decoding key 권장)
    odp_service_key: str

    # (선택) 기간/항공사
    odp_start_yyyymm: str = "202201"
    odp_end_yyyymm: Optional[str] = None
    airlines: Optional[Dict[str, str]] = None

    # AirPortal 기간
    airportal_start_year: int = 2015
    airportal_start_month: int = 1
    airportal_end_year: int = 2025
    airportal_end_month: int = 12

    # KAC 기간
    kac_start_yyyymm: str = "202001"
    kac_end_yyyymm: str = "202512"

    # (옵션) BOK ECOS API key
    bok_api_key: Optional[str] = None

    # (옵션) FDR 포함 여부
    include_fdr: bool = False
    fdr_start: str = "2010-01-01"


def collect_air_tourism_long(cfg: AirTourismCollectorConfig) -> pd.DataFrame:
    """
    3개(또는 5개) 소스를 모두 모아 하나의 long-format df로 반환
    """
    parts: List[pd.DataFrame] = []

    # 1) ODP Incheon airline stats
    odp_client = ODPIncheonAirlineStatsClient(
        ODPIncheonAirlineStatsConfig(
            service_key=cfg.odp_service_key,
            start_yyyymm=cfg.odp_start_yyyymm,
            end_yyyymm=cfg.odp_end_yyyymm,
            airlines=cfg.airlines,
        )
    )
    df_odp_raw = odp_client.fetch()
    df_odp_long = odp_client.to_long(df_odp_raw)
    parts.append(df_odp_long)

    # 2) AirPortal
    ap_client = AirPortalClient(
        AirPortalConfig(
            start_year=cfg.airportal_start_year,
            start_month=cfg.airportal_start_month,
            end_year=cfg.airportal_end_year,
            end_month=cfg.airportal_end_month,
        )
    )
    df_ap_raw = ap_client.download_period()
    df_ap_long = ap_client.to_long(df_ap_raw)
    parts.append(df_ap_long)

    # 3) KAC
    kac_client = KACClient(
        KACConfig(
            start_yyyymm=cfg.kac_start_yyyymm,
            end_yyyymm=cfg.kac_end_yyyymm,
        )
    )
    df_kac_raw = kac_client.fetch()
    df_kac_long = kac_client.to_long(df_kac_raw)
    parts.append(df_kac_long)

    # 4) (옵션) BOK ECOS CSI
    if cfg.bok_api_key:
        parts.append(fetch_travel_spending_expectation_csi_total(cfg.bok_api_key))

    # 5) (옵션) FDR
    if cfg.include_fdr:
        parts.append(fetch_oil_fx_fdr(cfg.fdr_start))

    # 결합
    out = pd.concat([p for p in parts if p is not None and not p.empty], ignore_index=True)

    # 기본 정리
    out["date"] = pd.to_datetime(out["date"], errors="coerce")
    out["value"] = pd.to_numeric(out["value"], errors="coerce")
    out = out.dropna(subset=["date"]).sort_values(["date", "indicator", "entity_type", "entity_code"]).reset_index(drop=True)

    # 중복 제거(핵심키 기준)
    out = out.drop_duplicates(subset=["date", "indicator", "source", "entity_type", "entity_code"], keep="last").reset_index(drop=True)
    return out



# =========================================================
# 실행 예시
# =========================================================
if __name__ == "__main__":
    # ⚠️ 사용 시 본인 KEY로 교체하세요.
    from DATA.KEYS import KEYS
    SERVICE_KEY = KEYS["ODPD"]
    # SERVICE_KEY = "YOUR_ODP_SERVICE_KEY"

    cfg = AirTourismCollectorConfig(
        odp_service_key=SERVICE_KEY,
        odp_start_yyyymm="202201",
        odp_end_yyyymm=None,
        airportal_start_year=2022,
        airportal_start_month=1,
        airportal_end_year=2025,
        airportal_end_month=12,
        kac_start_yyyymm="202201",
        kac_end_yyyymm="202512",
        bok_api_key=KEYS["BOK"],         # 예: KEYS["BOK"]
        include_fdr=True,
    )

    df_long = collect_air_tourism_long(cfg)
    print(df_long.tail(20))
    df_long.to_csv("air_tourism_long_V3.csv", index=False, encoding="utf-8-sig")
    print("[DONE] saved: air_tourism_long.csv")


C:\Users\82108\AppData\Local\Programs\Python\Python39\lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


[AIRPORTAL] ✓ 2022-01 rows=96


C:\Users\82108\AppData\Local\Programs\Python\Python39\lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


[AIRPORTAL] ✓ 2022-02 rows=95


C:\Users\82108\AppData\Local\Programs\Python\Python39\lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


...

[AIRPORTAL] ✓ 2025-10 rows=121


C:\Users\82108\AppData\Local\Programs\Python\Python39\lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


[AIRPORTAL] ✓ 2025-11 rows=113


C:\Users\82108\AppData\Local\Programs\Python\Python39\lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
C:\Users\82108\AppData\Local\Temp\ipykernel_9320\2599044318.py:189: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  for r in m[extras].to_dict(orient="records")


[AIRPORTAL] ✓ 2025-12 rows=113
[KAC] fetch 202201
[KAC] fetch 202202
[KAC] fetch 202203
[KAC] fetch 202204
[KAC] fetch 202205
[KAC] fetch 202206
[KAC] fetch 202207
[KAC] fetch 202208
[KAC] fetch 202209
[KAC] fetch 202210
[KAC] fetch 202211
[KAC] fetch 202212
[KAC] fetch 202301
[KAC] fetch 202302
[KAC] fetch 202303
[KAC] fetch 202304
[KAC] fetch 202305
[KAC] fetch 202306
[KAC] fetch 202307
[KAC] fetch 202308
[KAC] fetch 202309
[KAC] fetch 202310
[KAC] fetch 202311
[KAC] fetch 202312
[KAC] fetch 202401
[KAC] fetch 202402
[KAC] fetch 202403
[KAC] fetch 202404
[KAC] fetch 202405
[KAC] fetch 202406
[KAC] fetch 202407
[KAC] fetch 202408
[KAC] fetch 202409
[KAC] fetch 202410
[KAC] fetch 202411
[KAC] fetch 202412
[KAC] fetch 202501
[KAC] fetch 202502
[KAC] fetch 202503
[KAC] fetch 202504
[KAC] fetch 202505
[KAC] fetch 202506
[KAC] fetch 202507
[KAC] fetch 202508
[KAC] fetch 202509
[KAC] fetch 202510
[KAC] fetch 202511
[KAC] fetch 202512
            date                     indicator         va

In [2]:
# df_long.to_csv("air_tourism_long_V3.csv", index=False, encoding="utf-8-sig")

In [4]:
from simple_transform import long_to_wide, extract_series, get_mapping_table

print("=" * 80)
print("예시: 일별 데이터 자동 월말 변환")
print("=" * 80)

# CSV 로드
# df_long = pd.read_csv("air_tourism_long_V3.csv")
print(f"Long format: {len(df_long):,}행 로드")

# ==========================================
# 방법 1: 전체 Wide Format 변환 (기본 - 자동 리샘플링)
# ==========================================
print("\n[방법 1] 전체 변환 - 일별 데이터 자동 월말 변환")
df_wide = long_to_wide(df_long)  # resample_daily_to_monthly=True (기본값)

print(f"\nShape: {df_wide.shape}")
print(f"\n컬럼 목록:")
for i, col in enumerate(df_wide.columns, 1):
    print(f"  {i:2d}. {col}")

print(f"\n최근 10개월:")
print(df_wide.tail(10))

# CSV 저장
# df_wide.to_csv("air_tourism_wide.csv", encoding='utf-8-sig')
print(f"\n✅ air_tourism_wide.csv 저장 완료")


# ==========================================
# 방법 2: 리샘플링 하지 않고 원본 그대로 (옵션)
# ==========================================
print("\n" + "=" * 80)
print("[방법 2] 리샘플링 없이 원본 데이터 그대로 사용")

df_wide_raw = long_to_wide(df_long, resample_daily_to_monthly=False)
print(f"\nShape: {df_wide_raw.shape}")
print(f"날짜 개수: {len(df_wide_raw):,}개 (일별 포함)")


# ==========================================
# 방법 3: 특정 시계열만 추출
# ==========================================
print("\n" + "=" * 80)
print("[방법 3] 특정 시계열 추출")

# 3-1. 국제유가 (자동 월말 변환)
print("\n[WTI 유가 - 월말 기준]")
oil_wti = extract_series(df_long, 'oil_wti', 'FDR')  # resample_to_monthly=True (기본값)
print(f"최근 10개월:\n{oil_wti.tail(10)}\n")

# 3-2. 환율 (자동 월말 변환)
print("[USD/KRW 환율 - 월말 기준]")
usdkrw = extract_series(df_long, 'usdkrw', 'FDR')
print(f"최근 10개월:\n{usdkrw.tail(10)}\n")

# 3-3. 브렌트유 (자동 월말 변환)
print("[브렌트유 - 월말 기준]")
oil_brent = extract_series(df_long, 'oil_brent', 'FDR')
print(f"최근 10개월:\n{oil_brent.tail(10)}\n")

# 3-4. 항공사 데이터 (원래 월별)
print("[대한항공 여객수 - 인천공항]")
ke_pax = extract_series(df_long, 'icn_airline_pax_passenger', 'ODP_B551177', 'KE')
print(f"최근 10개월:\n{ke_pax.tail(10)}\n")


# ==========================================
# 방법 4: 선택한 지표만 모아서 DataFrame
# ==========================================
print("\n" + "=" * 80)
print("[방법 4] 경제 지표 + 항공사 데이터 조합")

df_selected = pd.DataFrame({
    'WTI유가': extract_series(df_long, 'oil_wti', 'FDR'),
    '브렌트유': extract_series(df_long, 'oil_brent', 'FDR'),
    '환율': extract_series(df_long, 'usdkrw', 'FDR'),
    '여행비CSI': extract_series(df_long, 'csi_travel_spending_expectation_total', 'BOK_ECOS'),
    '대한항공': extract_series(df_long, 'icn_airline_pax_passenger', 'ODP_B551177', 'KE'),
    '아시아나': extract_series(df_long, 'icn_airline_pax_passenger', 'ODP_B551177', 'OZ'),
    '제주항공': extract_series(df_long, 'icn_airline_pax_passenger', 'ODP_B551177', '7C'),
})

print(f"\nShape: {df_selected.shape}")
print(f"\n최근 12개월:")
print(df_selected.tail(12))

# 저장
# df_selected.to_csv("air_tourism_selected.csv", encoding='utf-8-sig')
print(f"\n✅ air_tourism_selected.csv 저장 완료")


# ==========================================
# 방법 5: 일별 데이터를 원본 그대로 유지하고 싶을 때
# ==========================================
print("\n" + "=" * 80)
print("[방법 5] 일별 데이터 원본 그대로 추출")

# resample_to_monthly=False 옵션 사용
oil_wti_daily = extract_series(df_long, 'oil_wti', 'FDR', resample_to_monthly=False)
print(f"\n일별 WTI 유가:")
print(f"총 {len(oil_wti_daily):,}개 (일별)")
print(f"최근 10일:\n{oil_wti_daily.tail(10)}")


# ==========================================
# 방법 6: 결측치 처리
# ==========================================
print("\n" + "=" * 80)
print("[방법 6] 결측치 확인 및 처리")

print(f"\n결측치 개수:")
null_counts = df_wide.isnull().sum()
for col in df_wide.columns:
    count = null_counts[col]
    if count > 0:
        pct = count / len(df_wide) * 100
        print(f"  {col}: {count}개 ({pct:.1f}%)")

# 선형 보간
df_filled = df_wide.interpolate(method='linear')
print(f"\n선형 보간 후 결측치: {df_filled.isnull().sum().sum()}개")

# 이전값으로 채우기
df_ffill = df_wide.fillna(method='ffill')
print(f"이전값 채우기 후 결측치: {df_ffill.isnull().sum().sum()}개")


# ==========================================
# 방법 7: 특정 기간만 필터링
# ==========================================
print("\n" + "=" * 80)
print("[방법 7] 특정 기간 데이터만 추출")

# 2020년 이후
df_2020 = df_wide[df_wide.index >= '2020-01-01']
print(f"\n2020년 이후: {len(df_2020):,}개월")

# 특정 기간 (2022-2024)
df_range = df_wide[(df_wide.index >= '2022-01-01') & (df_wide.index < '2025-01-01')]
print(f"2022-2024년: {len(df_range):,}개월")


# ==========================================
# 방법 8: 매핑 테이블 확인 및 수정
# ==========================================
print("\n" + "=" * 80)
print("[방법 8] 매핑 테이블 확인")

mapping = get_mapping_table()
print(f"\n전체 매핑 ({len(mapping)}개):")
print(mapping)

# FDR 데이터 확인
print(f"\nFDR 데이터 (일별 → 월말 변환):")
print(mapping[mapping['source'] == 'FDR'])


print("\n" + "=" * 80)
print("✅ 모든 예시 완료!")
print("\n생성된 파일:")
print("  - air_tourism_wide.csv (전체 데이터, 월말 기준)")
print("  - air_tourism_selected.csv (선택 지표)")
print("=" * 80)

예시: 일별 데이터 자동 월말 변환
Long format: 59,710행 로드

[방법 1] 전체 변환 - 일별 데이터 자동 월말 변환
일별 데이터 발견: 12,314행 → 월말 기준으로 리샘플링
리샘플링 결과: 585행 (월말 기준)
매핑 전 데이터: 47,981행
매핑 후 데이터: 1,516행
변환 결과: 210행 x 19열
날짜 범위: 2008-09-30 00:00:00 ~ 2026-02-28 00:00:00

Shape: (210, 19)

컬럼 목록:
   1. 국제유가_WTI
   2. 국제유가_브렌트
   3. 대한항공_공급석
   4. 대한항공_여객자수_인천공항
   5. 대한항공_여객자수_입국_공항공사
   6. 대한항공_여객자수_출국_공항공사
   7. 대한항공_화물_톤
   8. 아시아나_공급석
   9. 아시아나_여객자수_인천공항
  10. 아시아나_여객자수_입국_공항공사
  11. 아시아나_여객자수_출국_공항공사
  12. 아시아나_화물_톤
  13. 여행비지출전망_CSI
  14. 제주항공_공급석
  15. 제주항공_여객자수_인천공항
  16. 제주항공_여객자수_입국_공항공사
  17. 제주항공_여객자수_출국_공항공사
  18. 제주항공_화물_톤
  19. 환율

최근 10개월:
columns_name   국제유가_WTI   국제유가_브렌트   대한항공_공급석  대한항공_여객자수_인천공항  \
date                                                            
2025-05-31    60.790001  63.900002  3134113.0       1464134.0   
2025-06-30    65.110001  67.610001  3097192.0       1420141.0   
2025-07-31    69.260002  72.529999  3221505.0       1429416.0   
2025-08-31    64.010002  68.120003  3311492.0   

C:\Users\82108\AppData\Local\Temp\ipykernel_9320\238301319.py:125: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_ffill = df_wide.fillna(method='ffill')


In [5]:
df_selected.loc['2022':]

,WTI유가,브렌트유,환율,여행비CSI,대한항공,아시아나,제주항공
date,,,,,,,
2022-01-31,88.150002,91.209999,1207.709961,87.0,110373.0,68863.0,3411.0
2022-02-28,95.720001,100.989998,1197.589966,89.0,95082.0,66583.0,5320.0
2022-03-31,100.279999,107.910004,1214.500000,93.0,135224.0,91139.0,6883.0
2022-04-30,104.690002,109.339996,1272.290039,101.0,207207.0,153232.0,7995.0
2022-05-31,114.669998,122.839996,1241.739990,104.0,283988.0,211795.0,18938.0
2022-06-30,105.760002,114.809998,1287.709961,99.0,358968.0,253266.0,30175.0
2022-07-31,98.620003,110.010002,1302.280029,92.0,480411.0,308983.0,89015.0
2022-08-31,89.550003,96.489998,1342.260010,87.0,525309.0,344830.0,112207.0
2022-09-30,79.489998,87.959999,1430.170044,91.0,524011.0,330136.0,84543.0


In [6]:
import numpy as np

def add_derived_metrics(df_wide: pd.DataFrame) -> pd.DataFrame:
    """
    Wide format 데이터에 파생 지표 추가

    Parameters:
    -----------
    df_wide : pd.DataFrame
        Wide format 데이터 (index=date, columns=지표명)

    Returns:
    --------
    pd.DataFrame
        파생 지표가 추가된 DataFrame

    추가되는 컬럼:
    - 대한항공_여객자수: 인천공항 + 입국 + 출국
    - 아시아나_여객자수: 인천공항 + 입국 + 출국
    - 제주항공_여객자수: 인천공항 + 입국 + 출국
    - 유가_MoM: WTI 유가 전월대비 변화율
    - 환율_MoM: 환율 전월대비 변화율
    - 대한항공_여객자수_MoM: 여객자수 전월대비 변화율
    - 아시아나_여객자수_MoM: 여객자수 전월대비 변화율
    - 제주항공_여객자수_MoM: 여객자수 전월대비 변화율
    - 대한항공_탑승률: 여객자수 / 공급석
    - 아시아나_탑승률: 여객자수 / 공급석
    - 제주항공_탑승률: 여객자수 / 공급석
    - 대한항공_화물_톤_MoM: 화물 전월대비 변화율
    - 아시아나_화물_톤_MoM: 화물 전월대비 변화율
    """
    # 복사본 생성
    df = df_wide.copy()

    print("=" * 80)
    print("추가 분석 지표 계산 시작")
    print("=" * 80)

    # ========================================
    # 1. 여객자수 합산 (인천공항 + 입국 + 출국)
    # ========================================
    print("\n[1] 여객자수 합산")

    # 대한항공
    if all(col in df.columns for col in ['대한항공_여객자수_인천공항', '대한항공_여객자수_입국_공항공사', '대한항공_여객자수_출국_공항공사']):
        df['대한항공_여객자수'] = (
            df['대한항공_여객자수_인천공항'].fillna(0) +
            df['대한항공_여객자수_입국_공항공사'].fillna(0) +
            df['대한항공_여객자수_출국_공항공사'].fillna(0)
        )
        # 모두 NaN인 경우는 NaN으로
        mask = (df['대한항공_여객자수_인천공항'].isna() &
                df['대한항공_여객자수_입국_공항공사'].isna() &
                df['대한항공_여객자수_출국_공항공사'].isna())
        df.loc[mask, '대한항공_여객자수'] = np.nan
        print(f"  ✓ 대한항공_여객자수 생성 ({df['대한항공_여객자수'].notna().sum()}개 값)")
    else:
        print("  ⚠ 대한항공 여객자수 컬럼이 부족합니다")

    # 아시아나
    if all(col in df.columns for col in ['아시아나_여객자수_인천공항', '아시아나_여객자수_입국_공항공사', '아시아나_여객자수_출국_공항공사']):
        df['아시아나_여객자수'] = (
            df['아시아나_여객자수_인천공항'].fillna(0) +
            df['아시아나_여객자수_입국_공항공사'].fillna(0) +
            df['아시아나_여객자수_출국_공항공사'].fillna(0)
        )
        mask = (df['아시아나_여객자수_인천공항'].isna() &
                df['아시아나_여객자수_입국_공항공사'].isna() &
                df['아시아나_여객자수_출국_공항공사'].isna())
        df.loc[mask, '아시아나_여객자수'] = np.nan
        print(f"  ✓ 아시아나_여객자수 생성 ({df['아시아나_여객자수'].notna().sum()}개 값)")
    else:
        print("  ⚠ 아시아나 여객자수 컬럼이 부족합니다")

    # 제주항공
    if all(col in df.columns for col in ['제주항공_여객자수_인천공항', '제주항공_여객자수_입국_공항공사', '제주항공_여객자수_출국_공항공사']):
        df['제주항공_여객자수'] = (
            df['제주항공_여객자수_인천공항'].fillna(0) +
            df['제주항공_여객자수_입국_공항공사'].fillna(0) +
            df['제주항공_여객자수_출국_공항공사'].fillna(0)
        )
        mask = (df['제주항공_여객자수_인천공항'].isna() &
                df['제주항공_여객자수_입국_공항공사'].isna() &
                df['제주항공_여객자수_출국_공항공사'].isna())
        df.loc[mask, '제주항공_여객자수'] = np.nan
        print(f"  ✓ 제주항공_여객자수 생성 ({df['제주항공_여객자수'].notna().sum()}개 값)")
    else:
        print("  ⚠ 제주항공 여객자수 컬럼이 부족합니다")

    # ========================================
    # 2. MoM (Month-over-Month) 변화율 계산
    # ========================================
    print("\n[2] MoM 변화율 계산 (전월대비 % 변화)")

    mom_columns = {
        '국제유가_WTI': '유가_MoM',
        '환율': '환율_MoM',
        '대한항공_여객자수': '대한항공_여객자수_MoM',
        '아시아나_여객자수': '아시아나_여객자수_MoM',
        '제주항공_여객자수': '제주항공_여객자수_MoM',
        '대한항공_화물_톤': '대한항공_화물_톤_MoM',
        '아시아나_화물_톤': '아시아나_화물_톤_MoM',
    }

    for source_col, target_col in mom_columns.items():
        if source_col in df.columns:
            # MoM = (현재값 - 이전값) / 이전값 * 100
            df[target_col] = df[source_col].pct_change(fill_method=None) * 100
            non_null_count = df[target_col].notna().sum()
            print(f"  ✓ {target_col} 생성 ({non_null_count}개 값)")
        else:
            print(f"  ⚠ {source_col} 컬럼이 없어 {target_col} 생성 불가")

    # ========================================
    # 3. 탑승률 계산 (여객자수 / 공급석 * 100)
    # ========================================
    print("\n[3] 탑승률 계산 (여객자수 / 공급석 * 100)")

    load_factor_pairs = [
        ('대한항공_여객자수', '대한항공_공급석', '대한항공_탑승률'),
        ('아시아나_여객자수', '아시아나_공급석', '아시아나_탑승률'),
        ('제주항공_여객자수', '제주항공_공급석', '제주항공_탑승률'),
    ]

    for pax_col, supply_col, load_factor_col in load_factor_pairs:
        if pax_col in df.columns and supply_col in df.columns:
            df[load_factor_col] = (df[pax_col] / df[supply_col]) * 100
            non_null_count = df[load_factor_col].notna().sum()
            avg_load_factor = df[load_factor_col].mean()
            print(f"  ✓ {load_factor_col} 생성 ({non_null_count}개 값, 평균 {avg_load_factor:.1f}%)")
        else:
            print(f"  ⚠ {pax_col} 또는 {supply_col} 컬럼이 없어 {load_factor_col} 생성 불가")

    # ========================================
    # 결과 요약
    # ========================================
    print("\n" + "=" * 80)
    print("추가 분석 완료")
    print("=" * 80)

    added_columns = []
    for col in df.columns:
        if col not in df_wide.columns:
            added_columns.append(col)

    print(f"\n추가된 컬럼 ({len(added_columns)}개):")
    for i, col in enumerate(added_columns, 1):
        non_null = df[col].notna().sum()
        print(f"  {i:2d}. {col} ({non_null}개 값)")

    return df


def analyze_and_export(df_wide: pd.DataFrame, output_prefix: str = "air_tourism_analyzed"):
    """
    분석 수행 및 결과 저장

    Parameters:
    -----------
    df_wide : pd.DataFrame
        원본 Wide format 데이터
    output_prefix : str
        출력 파일명 접두사
    """
    # 파생 지표 추가
    df_analyzed = add_derived_metrics(df_wide)

    # 전체 데이터 저장
    full_output = f"{output_prefix}_전체.csv"
    df_analyzed.to_csv(full_output, encoding='utf-8-sig')
    print(f"\n✅ {full_output} 저장 완료")

    # 주요 지표만 선택하여 저장
    key_columns = [
        '국제유가_WTI', '유가_MoM',
        '환율', '환율_MoM',
        '여행비지출전망_CSI',
        '대한항공_여객자수', '대한항공_여객자수_MoM', '대한항공_탑승률', '대한항공_화물_톤', '대한항공_화물_톤_MoM',
        '아시아나_여객자수', '아시아나_여객자수_MoM', '아시아나_탑승률', '아시아나_화물_톤', '아시아나_화물_톤_MoM',
        '제주항공_여객자수', '제주항공_여객자수_MoM', '제주항공_탑승률', '제주항공_화물_톤',
    ]

    available_key_columns = [col for col in key_columns if col in df_analyzed.columns]
    df_key = df_analyzed[available_key_columns]

    key_output = f"{output_prefix}_주요지표.csv"
    df_key.to_csv(key_output, encoding='utf-8-sig')
    print(f"✅ {key_output} 저장 완료")

    # 기초 통계
    print(f"\n[기초 통계]")
    print(df_key.describe())

    return df_analyzed


# ==========================================
# 사용 예시
# ==========================================
if __name__ == "__main__":
    import sys
    sys.path.insert(0, '/home/claude')
    from simple_transform import long_to_wide

    print("=" * 80)
    print("Air Tourism 추가 분석 실행")
    print("=" * 80)

    # 데이터 로드 및 변환
    df_long = pd.read_csv(r"C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\ODP\Industry_Data_Process\2_Air_Tourism\air_tourism_long_V3.csv")
    df_wide = long_to_wide(df_long)

    # 추가 분석 수행
    df_analyzed = analyze_and_export(df_wide, output_prefix="air_tourism_analyzed")

    # 특정 기간만 필터링 (예: 2020년 이후)
    df_range = df_analyzed[df_analyzed.index >= '2020-01-01']

    print(f"\n[특정 기간 분석: 2020년 이후]")
    print(f"기간: {df_range.index.min()} ~ {df_range.index.max()}")
    print(f"Shape: {df_range.shape}")
    print(f"\n최근 10개월:")
    print(df_range.tail(10))

    # 기간별 저장
    df_range.to_csv("air_tourism_analyzed_2020이후.csv", encoding='utf-8-sig')
    print(f"\n✅ air_tourism_analyzed_2020이후.csv 저장 완료")

    print("\n" + "=" * 80)
    print("✅ 모든 분석 완료!")
    print("=" * 80)



Air Tourism 추가 분석 실행
일별 데이터 발견: 12,314행 → 월말 기준으로 리샘플링
리샘플링 결과: 585행 (월말 기준)
매핑 전 데이터: 47,981행
매핑 후 데이터: 1,516행
변환 결과: 210행 x 19열
날짜 범위: 2008-09-30 00:00:00 ~ 2026-02-28 00:00:00
추가 분석 지표 계산 시작

[1] 여객자수 합산
  ✓ 대한항공_여객자수 생성 (49개 값)
  ✓ 아시아나_여객자수 생성 (49개 값)
  ✓ 제주항공_여객자수 생성 (49개 값)

[2] MoM 변화율 계산 (전월대비 % 변화)
  ✓ 유가_MoM 생성 (194개 값)
  ✓ 환율_MoM 생성 (194개 값)
  ✓ 대한항공_여객자수_MoM 생성 (48개 값)
  ✓ 아시아나_여객자수_MoM 생성 (48개 값)
  ✓ 제주항공_여객자수_MoM 생성 (48개 값)
  ✓ 대한항공_화물_톤_MoM 생성 (47개 값)
  ✓ 아시아나_화물_톤_MoM 생성 (47개 값)

[3] 탑승률 계산 (여객자수 / 공급석 * 100)
  ✓ 대한항공_탑승률 생성 (48개 값, 평균 83.1%)
  ✓ 아시아나_탑승률 생성 (48개 값, 평균 82.8%)
  ✓ 제주항공_탑승률 생성 (48개 값, 평균 90.1%)

추가 분석 완료

추가된 컬럼 (13개):
   1. 대한항공_여객자수 (49개 값)
   2. 아시아나_여객자수 (49개 값)
   3. 제주항공_여객자수 (49개 값)
   4. 유가_MoM (194개 값)
   5. 환율_MoM (194개 값)
   6. 대한항공_여객자수_MoM (48개 값)
   7. 아시아나_여객자수_MoM (48개 값)
   8. 제주항공_여객자수_MoM (48개 값)
   9. 대한항공_화물_톤_MoM (47개 값)
  10. 아시아나_화물_톤_MoM (47개 값)
  11. 대한항공_탑승률 (48개 값)
  12. 아시아나_탑승률 (48개 값)
  13. 제주항공_탑승률 (48개 값)

✅ air_tourism_a

In [7]:
df_range.loc['2022': '2025']

columns_name,국제유가_WTI,국제유가_브렌트,대한항공_공급석,대한항공_여객자수_인천공항,대한항공_여객자수_입국_공항공사,대한항공_여객자수_출국_공항공사,대한항공_화물_톤,아시아나_공급석,아시아나_여객자수_인천공항,아시아나_여객자수_입국_공항공사,...,유가_MoM,환율_MoM,대한항공_여객자수_MoM,아시아나_여객자수_MoM,제주항공_여객자수_MoM,대한항공_화물_톤_MoM,아시아나_화물_톤_MoM,대한항공_탑승률,아시아나_탑승률,제주항공_탑승률
date,,,,,,,,,,,,,,,,,,,,,
2022-01-31,88.150002,91.209999,1518424.0,110373.0,494729.0,494729.0,142430.0,1126542.0,68863.0,395059.0,...,17.205162,1.497613,NaN,NaN,NaN,NaN,NaN,72.432404,76.264888,90.161067
2022-02-28,95.720001,100.989998,1417109.0,95082.0,438878.0,438878.0,123226.0,1122835.0,66583.0,375899.0,...,8.587634,-0.837949,-11.546592,-4.745937,-6.837547,-13.483115,-13.393529,68.649483,72.885241,89.220313
2022-03-31,100.279999,107.910004,1402850.0,135224.0,370284.0,370284.0,150291.0,1137419.0,91139.0,325021.0,...,4.763892,1.412005,-9.975556,-9.433259,-12.309381,21.963709,17.019983,62.429483,65.163409,80.755073
2022-04-30,104.690002,109.339996,1582308.0,207207.0,550995.0,550995.0,144106.0,1235022.0,153232.0,431728.0,...,4.397690,4.758340,49.487207,37.171352,14.580148,-4.115350,-4.836291,82.739707,82.321449,92.770910
2022-05-31,114.669998,122.839996,1749528.0,283988.0,606774.0,606774.0,139944.0,1349825.0,211795.0,473588.0,...,9.532902,-2.401186,14.385841,14.019935,9.903070,-2.888152,0.784858,85.596572,85.879799,94.015169
2022-06-30,105.760002,114.809998,1751710.0,358968.0,599680.0,599694.0,134369.0,1336398.0,253266.0,459509.0,...,-7.770120,3.702061,4.060403,1.142744,-5.995861,-3.983736,-0.408726,88.961186,87.733894,93.650434
2022-07-31,98.620003,110.010002,1968732.0,480411.0,586348.0,586401.0,133453.0,1474623.0,308983.0,435680.0,...,-6.751134,1.131471,6.084544,0.676433,1.709624,-0.681705,-6.025464,83.970799,80.047917,91.584964
2022-08-31,89.550003,96.489998,2009007.0,525309.0,573948.0,573653.0,131768.0,1577661.0,344830.0,455394.0,...,-9.196917,3.069999,1.194682,6.362731,8.654890,-1.262617,5.721990,83.270491,79.580531,92.196151
2022-09-30,79.489998,87.959999,1941593.0,524011.0,501451.0,501680.0,127637.0,1445439.0,330136.0,396797.0,...,-11.233953,6.549404,-8.713439,-10.487124,-16.866567,-3.135056,1.079643,78.654074,77.751050,86.462605


In [12]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mtick
from matplotlib.gridspec import GridSpec
from typing import Optional, List, Tuple

# =========================================================
# 0) 한글 폰트 자동 설정
# =========================================================
def set_korean_font():
    import matplotlib
    import matplotlib.font_manager as fm
    matplotlib.rcParams["axes.unicode_minus"] = False
    fonts = {f.name for f in fm.fontManager.ttflist}
    for cand in ["Malgun Gothic", "AppleGothic", "NanumGothic", "Noto Sans CJK KR", "Noto Sans KR"]:
        if cand in fonts:
            matplotlib.rcParams["font.family"] = cand
            return cand
    return None


# =========================================================
# 1) DF 정규화: date_col 없으면 index를 date_col로 생성
# =========================================================
def normalize_df_date(df: pd.DataFrame, date_col: str = "date") -> pd.DataFrame:
    out = df.copy()

    # date_col이 컬럼에 없으면 index를 날짜로 사용
    if date_col not in out.columns:
        idx = out.index

        # index가 datetime이 아니면 변환 시도
        if not pd.api.types.is_datetime64_any_dtype(idx):
            try:
                idx = pd.to_datetime(idx)
            except Exception as e:
                raise ValueError(
                    f"date_col='{date_col}' 컬럼이 없고, index를 datetime으로 변환도 실패했습니다. "
                    f"(index dtype={out.index.dtype})"
                ) from e

        out = out.copy()
        out[date_col] = idx
    else:
        out[date_col] = pd.to_datetime(out[date_col])

    out = out.sort_values(date_col)
    return out


def has_data(df: pd.DataFrame, col: str) -> bool:
    return (col in df.columns) and df[col].notna().any()


def compute_yoy(series: pd.Series) -> pd.Series:
    return series.pct_change(12, fill_method=None) * 100.0


def format_time_axis(ax):
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y/%m"))
    for lab in ax.get_xticklabels():
        lab.set_fontsize(8)


def format_y_commas(ax):
    ax.yaxis.set_major_formatter(mtick.StrMethodFormatter("{x:,.0f}"))


def plot_series(ax, x, y, title, kind="line", percent=False):
    if kind == "bar":
        ax.bar(x, y, width=20)
    else:
        ax.plot(x, y, lw=1.5)

    ax.set_title(title, fontsize=10)
    ax.grid(True, alpha=0.3)
    format_time_axis(ax)

    if percent:
        ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    else:
        format_y_commas(ax)


# =========================================================
# 2) 섹션 헤더 (링크 박스 제거)
# =========================================================
def add_section_header(fig, gs_cell, number: int, title: str):
    ax = fig.add_subplot(gs_cell)
    ax.axis("off")

    ax.add_patch(plt.Rectangle((0, 0), 1, 1, transform=ax.transAxes,
                               facecolor="#f2f2f2", edgecolor="black", lw=0.6))

    ax.add_patch(plt.Rectangle((0, 0), 0.06, 1, transform=ax.transAxes,
                               facecolor="#d7191c", edgecolor="black", lw=0.6))
    ax.text(0.03, 0.5, str(number), va="center", ha="center",
            color="white", fontsize=13, fontweight="bold", transform=ax.transAxes)

    ax.text(0.08, 0.5, title, va="center", ha="left",
            fontsize=13, fontweight="bold", transform=ax.transAxes)


# =========================================================
# 3) 대시보드 생성 (Python 3.9 호환)
# =========================================================
ChartSpec = Tuple[str, str, str, bool]               # (title, col, kind, percent)
SectionSpec = Tuple[int, str, List[ChartSpec]]       # (number, section_title, charts)

def make_dashboard(df: pd.DataFrame,
                   outpath: str,
                   date_col: str = "date",
                   sections: Optional[List[SectionSpec]] = None,
                   suptitle: str = "대시보드"):

    set_korean_font()

    # ★ 핵심: date_col 없으면 index를 date로 만들어서 사용
    df = normalize_df_date(df, date_col=date_col)

    if sections is None:
        sections = []

        # 1) 선행/유가/환율
        s1: List[ChartSpec] = []
        if has_data(df, "여행비지출전망_CSI"):
            s1.append(("여행비 지출전망CSI", "여행비지출전망_CSI", "line", False))
        if has_data(df, "국제유가_두바이유"):
            s1.append(("두바이유", "국제유가_두바이유", "line", False))
        elif has_data(df, "국제유가_브렌트"):
            s1.append(("브렌트유", "국제유가_브렌트", "line", False))
        elif has_data(df, "국제유가_WTI"):
            s1.append(("WTI", "국제유가_WTI", "line", False))
        if has_data(df, "환율"):
            s1.append(("환율 (기말기준)", "환율", "line", False))
        if s1:
            sections.append((1, "여행비 지출 전망 (선행지표), 유가 및 환율", s1))

        # 2) FSC (대한항공 중심)
        s2: List[ChartSpec] = []
        if has_data(df, "대한항공_여객자수"):
            s2.append(("대한항공 여객자수", "대한항공_여객자수", "bar", False))
        if has_data(df, "대한항공_화물"):
            s2.append(("대한항공 화물", "대한항공_화물", "bar", False))
            yoy = compute_yoy(df["대한항공_화물"])
            if yoy.notna().any():
                df["_대한항공_화물_YoY"] = yoy
                s2.append(("대한항공 화물 YoY", "_대한항공_화물_YoY", "line", True))

        if has_data(df, "FSC_탑승율"):
            s2.append(("FSC 탑승율", "FSC_탑승율", "line", True))
        elif has_data(df, "대한항공_탑승률"):
            s2.append(("FSC 탑승율", "대한항공_탑승률", "line", True))

        if has_data(df, "FSC_공급석"):
            s2.append(("FSC 공급석", "FSC_공급석", "line", False))
        elif has_data(df, "대한항공_공급석"):
            s2.append(("FSC 공급석", "대한항공_공급석", "line", False))

        if s2:
            sections.append((2, "FSC 승객, 이용자수, 공급석(탑승율), 화물수송", s2))

        # 3) LCC (제주항공 중심)
        s3: List[ChartSpec] = []
        if has_data(df, "제주항공_여객자수"):
            s3.append(("제주항공 여객자수", "제주항공_여객자수", "bar", False))

        if has_data(df, "LCC_탑승율"):
            s3.append(("LCC 탑승율", "LCC_탑승율", "line", True))
        elif has_data(df, "제주항공_탑승률"):
            s3.append(("LCC 탑승율", "제주항공_탑승률", "line", True))

        if has_data(df, "LCC_공급석"):
            s3.append(("LCC 공급석", "LCC_공급석", "line", False))
        elif has_data(df, "제주항공_공급석"):
            s3.append(("LCC 공급석", "제주항공_공급석", "line", False))

        if has_data(df, "제주항공_화물"):
            s3.append(("제주항공 화물", "제주항공_화물", "bar", False))
            yoy = compute_yoy(df["제주항공_화물"])
            if yoy.notna().any():
                df["_제주항공_화물_YoY"] = yoy
                s3.append(("제주항공 화물 YoY", "_제주항공_화물_YoY", "line", True))

        if s3:
            sections.append((3, "LCC 승객, 이용자수, 공급석(탑승율), 화물수송", s3))

    if not sections:
        raise ValueError("표시할 데이터가 없습니다(컬럼 없음 또는 전부 NaN).")

    fig_h = 6.5 * len(sections)
    fig = plt.figure(figsize=(18, fig_h), constrained_layout=False)

    outer = GridSpec(len(sections), 1, figure=fig, hspace=0.45)

    for i, (num, title, charts) in enumerate(sections):
        # 헤더-차트 간격 2배
        inner = outer[i].subgridspec(2, 1, height_ratios=[0.14, 0.86], hspace=0.16)

        add_section_header(fig, inner[0], num, title)

        k = len(charts)
        cols = min(5, max(3, k))
        rows = int(np.ceil(k / cols))
        chart_gs = inner[1].subgridspec(rows, cols, wspace=0.25, hspace=0.35)

        for j, (ctitle, col, kind, percent) in enumerate(charts):
            r, c = divmod(j, cols)
            ax = fig.add_subplot(chart_gs[r, c])

            if (col not in df.columns) or (not df[col].notna().any()):
                ax.axis("off")
                continue

            plot_series(ax, df[date_col], df[col], ctitle, kind=kind, percent=percent)

        for j in range(k, rows * cols):
            ax = fig.add_subplot(chart_gs[j // cols, j % cols])
            ax.axis("off")

    fig.suptitle(suptitle, fontsize=14, fontweight="bold", y=0.995)
    fig.tight_layout(rect=[0, 0, 1, 0.99])

    os.makedirs(os.path.dirname(outpath) or ".", exist_ok=True)
    fig.savefig(outpath, dpi=200)
    plt.close(fig)


In [14]:
make_dashboard(
    df=df_range,                      # <- 이미 만들어진 DF
    outpath="air_tourism_dashboard.png",
    date_col="columns_name",                  # DF에 없으면 index에서 자동 생성됨
    sections=None,
    suptitle="항공/관광 지표 대시보드"
)

C:\Users\82108\AppData\Local\Temp\ipykernel_9320\4169397389.py:226: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout(rect=[0, 0, 1, 0.99])


### 속도 상승을 위한 batch 저장